# f_compare

Run all cells. Outputs are written to this task's `output/` folder.


In [1]:
%run ~/Desktop/SHL_dblp_comparable/common/core.ipynb


/Users/slmagid/miniforge3/envs/research313/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [2]:
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
ROOT = Path.home() / 'Desktop' / 'SHL_dblp_comparable'
CONFIG = read_config(ROOT)
AARC = Path.home() / 'Desktop' / 'SHL_aarc_modeling'
AFIT = AARC / 'd_fit' / 'output' / 'results'
APLOT = AARC / 'f_plot' / 'output' / 'results'
FIT = ROOT / 'd_fit' / 'output' / 'results'
PREP = ROOT / 'c_prepare' / 'output'
SIM = ROOT / 'e_simulate' / 'output' / 'trajectories'
OUTPUT = ROOT / 'f_compare' / 'output'
FIG = OUTPUT / 'figures'
RESULTS = OUTPUT / 'results'
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
FIG.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
required = [APLOT / 'memory_kernel_summary.csv', APLOT / 'raw_memory_coefficients.csv', AARC / 'd_fit' / 'output' / 'analysis_manifest.json']
if any((not p.exists() for p in required)):
    raise FileNotFoundError('Run SHL_aarc_modeling through f_plot before cross-source comparison')
am = json.loads((AARC / 'd_fit' / 'output' / 'analysis_manifest.json').read_text())
dm = json.loads((ROOT / 'd_fit' / 'output' / 'analysis_manifest.json').read_text())
for key in ('max_observed_years', 'ar_order', 'memory_lags', 'memory_stages', 'raw_ar_lags', 'ar_lag_preconditioning', 'early_age_nesting_interactions', 'stage_duration_averaging', 'core_sha256'):
    if am.get(key) != dm.get(key):
        raise ValueError(f'Design mismatch for {key}: AARC={am.get(key)} DBLP={dm.get(key)}')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.22, 'lines.linewidth': 2})
palette = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9', '#332288']
styles = ['-', '--', ':', '-.', (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1))]
markers = ['o', 's', '^', 'D', 'P', 'v', 'X']
aparams = pd.read_csv(AFIT / 'stage_parameters.csv')
araw = pd.read_csv(APLOT / 'raw_memory_coefficients.csv')
asummary = pd.read_csv(APLOT / 'memory_kernel_summary.csv')
dparams = pd.read_csv(FIT / 'stage_parameters.csv')
dboot = pd.read_csv(FIT / 'bootstrap_stage_parameters.csv.gz')
cohorts = sorted(dparams.analysis_group.unique())
primary = 'all_eligible_cropped13'
summary = []
raw_parts = []
for cohort in cohorts:
    for equation in ('activity', 'positive'):
        p = coefficient_points(dparams, cohort, equation)
        b = coefficient_points(dboot, cohort, equation)
        ci = b.groupby(['stage', 'lag']).weight.quantile([0.025, 0.5, 0.975]).unstack().rename(columns={0.025: 'ci_low', 0.5: 'bootstrap_median', 0.975: 'ci_high'}).reset_index()
        p = p.merge(ci, on=['stage', 'lag'], how='left')
        raw_parts.append(p)
        full = fit_shared_shape(p, 'geometric')
        rhos = []
        for _, bp in b.groupby('replicate'):
            r = fit_shared_shape(bp, 'geometric')['rho']
            if np.isfinite(r):
                rhos.append(r)
        lo, med, hi = np.quantile(rhos, [0.025, 0.5, 0.975]) if rhos else (np.nan, np.nan, np.nan)
        weights = p[p.stage.isin(MEMORY_STAGES)].weight.to_numpy()
        summary.append({'analysis_group': cohort, 'source': 'DBLP', 'equation': equation, **full, 'rho_boot_median': med, 'rho_ci_low': lo, 'rho_ci_high': hi, 'positive_weight_fraction': float(np.mean(weights > 0))})
dsummary = pd.DataFrame(summary)
draw = pd.concat(raw_parts, ignore_index=True)
dsummary.to_csv(RESULTS / 'dblp_memory_kernel_summary.csv', index=False)
draw.to_csv(RESULTS / 'dblp_raw_memory_coefficients.csv', index=False)
ac = asummary.copy()
ac['source'] = 'AARC'
combined = pd.concat([ac, dsummary], ignore_index=True, sort=False)
combined.to_csv(RESULTS / 'combined_memory_kernel_summary.csv', index=False)
domains = sorted(ac.analysis_group.unique())
colors = {g: palette[i] for i, g in enumerate(domains)}
for equation in ('activity', 'positive'):
    a = ac[ac.equation.eq(equation)].set_index('analysis_group').reindex(domains).reset_index()
    d = dsummary[dsummary.equation.eq(equation) & dsummary.analysis_group.eq(primary)].iloc[0]
    labels = domains + ['DBLP CS — all eligible']
    fig, ax = plt.subplots(figsize=(9, 6))
    for i, row in a.iterrows():
        center = row.rho_boot_median if np.isfinite(row.rho_boot_median) else row.rho
        ax.errorbar(center, i, xerr=np.array([[max(center - row.rho_ci_low, 0)], [max(row.rho_ci_high - center, 0)]]), fmt=markers[i], color=colors[row.analysis_group], capsize=3)
        ax.scatter(row.rho, i, marker='|', s=70, color=colors[row.analysis_group])
    i = len(domains)
    center = d.rho_boot_median if np.isfinite(d.rho_boot_median) else d.rho
    ax.errorbar(center, i, xerr=np.array([[max(center - d.rho_ci_low, 0)], [max(d.rho_ci_high - center, 0)]]), fmt='D', ms=8, color='#111111', capsize=4, label='DBLP primary')
    ax.scatter(d.rho, i, marker='|', s=80, color='#111111')
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Implied geometric retention $\\hat{\\rho}_{shape}$')
    ax.set_title(f'13-year, raw-lag comparison — {equation} margin\npoints = bootstrap median; short vertical ticks = full-sample estimate', fontweight='bold')
    ax.legend(frameon=False)
    fig.savefig(FIG / f'combined_rho_forest__{equation}.png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
    c = dsummary[dsummary.equation.eq(equation)].copy()
    fig, ax = plt.subplots(figsize=(9, 4.8))
    y = np.arange(len(c))
    for i, row in c.reset_index(drop=True).iterrows():
        color = '#111111' if row.analysis_group == primary else '#777777'
        mark = 'D' if row.analysis_group == primary else 'o'
        center = row.rho_boot_median if np.isfinite(row.rho_boot_median) else row.rho
        ax.errorbar(center, i, xerr=np.array([[max(center - row.rho_ci_low, 0)], [max(row.rho_ci_high - center, 0)]]), fmt=mark, color=color, capsize=3)
        ax.scatter(row.rho, i, marker='|', s=70, color=color)
    ax.set_yticks(y, c.analysis_group.str.replace('_', ' '))
    ax.set_xlim(0, 1)
    ax.set_xlabel('$\\hat{\\rho}_{shape}$')
    ax.set_title(f'DBLP selection sensitivity — {equation} margin\npoints = bootstrap median; short vertical ticks = full-sample estimate', fontweight='bold')
    fig.savefig(FIG / f'dblp_cohort_sensitivity__{equation}.png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
for equation in ('activity', 'positive'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
    relevant = pd.concat([araw[araw.equation.eq(equation) & araw.stage.isin(MEMORY_STAGES)].assign(source='AARC'), draw[draw.equation.eq(equation) & draw.stage.isin(MEMORY_STAGES) & draw.analysis_group.eq(primary)].assign(source='DBLP')])
    pos = relevant.weight[relevant.weight > 0]
    floor = max(float(pos.min()) * 0.45 if len(pos) else 1e-05, 1e-05)
    for ax, stage in zip(axes, MEMORY_STAGES):
        for i, g in enumerate(domains):
            p = araw[araw.analysis_group.eq(g) & araw.equation.eq(equation) & araw.stage.eq(stage)].sort_values('lag')
            ax.plot(p.lag, p.weight.where(p.weight > 0, np.nan), color=colors[g], ls=styles[i], marker=markers[i], ms=4, label=f'AARC: {g}')
            bad = p.weight <= 0
            if bad.any():
                ax.scatter(p.loc[bad, 'lag'], np.full(bad.sum(), floor), marker='x', color=colors[g])
        p = draw[draw.analysis_group.eq(primary) & draw.equation.eq(equation) & draw.stage.eq(stage)].sort_values('lag')
        ax.plot(p.lag, p.weight.where(p.weight > 0, np.nan), color='#111111', lw=3, ls='--', marker='D', ms=6, label='DBLP CS: all eligible')
        bad = p.weight <= 0
        if bad.any():
            ax.scatter(p.loc[bad, 'lag'], np.full(bad.sum(), floor), marker='x', s=55, color='#111111')
        ax.set_yscale('log')
        ax.set_ylim(bottom=floor * 0.8)
        ax.set_xticks(MEMORY_LAGS)
        ax.set_xlabel('Historical lag k')
        ax.set_title(f'Stage {stage}')
    axes[0].set_ylabel('Positive raw AR coefficient (log scale)')
    axes[1].legend(ncol=2, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(f'AARC domains vs 13-year DBLP CS — {equation} margin (× = nonpositive)', fontweight='bold')
    fig.savefig(FIG / f'combined_raw_memory_logscale__{equation}.png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
asample = pd.read_csv(APLOT / 'sample_size_diagnostics.csv')
dlength = pd.read_csv(PREP / 'trajectory_lengths.csv')
dsupport = pd.read_csv(PREP / 'lag_support.csv')
dsize = []
for cohort in cohorts:
    n = dlength[dlength.analysis_group.eq(cohort)].person_domain_id.nunique()
    eff = dsupport[dsupport.analysis_group.eq(cohort) & dsupport.stage.isin(MEMORY_STAGES)].lag6_people.min()
    for equation in ('activity', 'positive'):
        row = dsummary[dsummary.analysis_group.eq(cohort) & dsummary.equation.eq(equation)].iloc[0]
        dsize.append({'analysis_group': cohort, 'equation': equation, 'person_domain_pairs': n, 'min_lag6_people': eff, 'rho': row.rho, 'rho_boot_median': row.rho_boot_median, 'rho_ci_low': row.rho_ci_low, 'rho_ci_high': row.rho_ci_high})
dsize = pd.DataFrame(dsize)
dsize.to_csv(RESULTS / 'dblp_sample_size_diagnostics.csv', index=False)
for equation in ('activity', 'positive'):
    fig, ax = plt.subplots(figsize=(9, 5))
    a = asample[asample.equation.eq(equation)]
    for i, row in a.reset_index(drop=True).iterrows():
        center = row.rho_boot_median if np.isfinite(row.rho_boot_median) else row.rho
        ax.errorbar(row.min_lag6_people, center, yerr=np.array([[max(center - row.rho_ci_low, 0)], [max(row.rho_ci_high - center, 0)]]), fmt=markers[i], color=colors[row.analysis_group], capsize=2)
        ax.annotate(row.analysis_group, (row.min_lag6_people, center), xytext=(4, 3), textcoords='offset points', fontsize=7)
    d = dsize[dsize.analysis_group.eq(primary) & dsize.equation.eq(equation)].iloc[0]
    center = d.rho_boot_median if np.isfinite(d.rho_boot_median) else d.rho
    ax.errorbar(d.min_lag6_people, center, yerr=np.array([[max(center - d.rho_ci_low, 0)], [max(d.rho_ci_high - center, 0)]]), fmt='D', ms=8, color='#111', capsize=4)
    ax.annotate('DBLP CS', (d.min_lag6_people, center), xytext=(5, -12), textcoords='offset points', fontweight='bold')
    ax.set(xlabel='Minimum people supporting lag 6 across memory stages', ylabel='$\\hat{\\rho}_{shape}$', ylim=(0, 1), title=f'Effective support and estimated retention — {equation} margin')
    fig.savefig(FIG / f'combined_rho_vs_effective_n__{equation}.png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
trans = pd.read_csv(PREP / 'modeling_transitions.csv.gz')
data = trans[trans.analysis_group.eq(primary) & trans.split.eq('test')]
people = sorted(data.person_domain_id.unique())
emp = np.full((len(people), 13), np.nan)
lookup = {p: i for i, p in enumerate(people)}
for row in data.itertuples(index=False):
    emp[lookup[row.person_domain_id], int(row.transition_age)] = row.q_prev
    emp[lookup[row.person_domain_id], int(row.target_age)] = row.q
matrices = {'Empirical': emp}
for path in SIM.glob(f'{slug(primary)}__*.npz'):
    z = np.load(path, allow_pickle=True)
    matrices[str(z['model'])] = z['q']

def stats(q):
    return pd.DataFrame([{'age': age, 'mean': np.nanmean(q[:, age]), 'log_variance': np.nanvar(np.log1p(q[:, age])), 'zero_fraction': np.nanmean(q[:, age] <= 0), 'rank_rho': 1.0 if age == 0 else spearmanr(q[np.isfinite(q[:, age]), 0], q[np.isfinite(q[:, age]), age]).statistic} for age in range(q.shape[1])])
if len(matrices) == 3:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    mstyle = {'Empirical': ('#111', '-', 'o'), MODEL_T5: ('#0072B2', '--', 's'), MODEL_AR: ('#D55E00', '-.', '^')}
    for label, q in matrices.items():
        s = stats(q)
        c, ls, m = mstyle[label]
        for ax, col in zip(axes.ravel(), ('mean', 'log_variance', 'zero_fraction', 'rank_rho')):
            ax.plot(s.age, s[col], label=label, color=c, ls=ls, marker=m, ms=3)
    for ax, title in zip(axes.ravel(), ('Mean productivity', 'Log variance', 'Inactivity', 'Rank persistence')):
        ax.set_title(title)
    axes[1, 0].set_xlabel('Career age')
    axes[1, 1].set_xlabel('Career age')
    axes[0, 1].legend(frameon=False)
    fig.suptitle('DBLP CS primary cohort: 13-year observation-matched simulation', fontweight='bold')
    fig.savefig(FIG / 'dblp_primary_dynamic_diagnostics.png', bbox_inches='tight', facecolor='white')
    plt.close(fig)
overlap = json.loads((PREP / 'overlap_report.json').read_text())
write_json(RESULTS / 'overlap_report.json', overlap)
report = {'status': 'passed', 'figures': len(list(FIG.glob('*.png'))), 'strict_design_match': True, 'primary_dblp_cohort': primary, 'aarc_domains': len(domains), 'dblp_cohorts': cohorts, 'overlap': overlap, 'interpretation': 'cross-sample/cross-domain replication using DBLP-linked productivity, not an independent measurement-source replication'}
write_json(OUTPUT / 'quality_report.json', report)
print(json.dumps(report, indent=2))


IndexError: list index out of range

In [ ]:
from matplotlib.lines import Line2D

aarc_raw = pd.read_csv(
    AARC / "f_plot" / "output" / "results" / "raw_memory_coefficients.csv"
)

dblp_raw = pd.read_csv(
    OUTPUT / "results" / "dblp_raw_memory_coefficients.csv"
)

dblp_raw = dblp_raw[
    dblp_raw.analysis_group.eq("all_eligible_cropped13")
].copy()

dblp_raw["analysis_group"] = "DBLP Computer Science"

plot_data = pd.concat(
    [aarc_raw, dblp_raw],
    ignore_index=True
)

plot_data = plot_data[
    plot_data.stage.isin(["5-7", "8-11"])
    & plot_data.lag.between(2, 6)
].copy()

stage_durations = {
    "5-7": 3,
    "8-11": 4,
}

plot_data["stage_duration"] = plot_data.stage.map(stage_durations)

averaged = (
    plot_data
    .assign(weighted_weight=lambda x: x.weight * x.stage_duration)
    .groupby(
        ["analysis_group", "equation", "lag"],
        as_index=False,
    )
    .agg(
        weighted_weight=("weighted_weight", "sum"),
        total_duration=("stage_duration", "sum"),
    )
)

averaged["weight"] = (
    averaged.weighted_weight
    / averaged.total_duration
)

fields = [
    "Applied Sciences",
    "Education",
    "Engineering",
    "Journalism, Media, Communication",
    "Mathematics and Computing",
    "Natural Sciences",
    "Social Sciences",
    "DBLP Computer Science",
]

colors = {
    "Applied Sciences": "#0072B2",
    "Education": "#D55E00",
    "Engineering": "#009E73",
    "Journalism, Media, Communication": "#CC79A7",
    "Mathematics and Computing": "#E69F00",
    "Natural Sciences": "#56B4E9",
    "Social Sciences": "#332288",
    "DBLP Computer Science": "#000000",
}

markers = {
    "Applied Sciences": "o",
    "Education": "s",
    "Engineering": "^",
    "Journalism, Media, Communication": "D",
    "Mathematics and Computing": "P",
    "Natural Sciences": "v",
    "Social Sciences": "X",
    "DBLP Computer Science": "*",
}

equation_titles = {
    "activity": "Activity margin",
    "positive": "Positive-output margin",
}

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
    sharex=True,
)

for ax, equation in zip(axes, ["activity", "positive"]):
    panel = averaged[
        averaged.equation.eq(equation)
    ].copy()

    positive_weights = panel.loc[
        panel.weight.gt(0),
        "weight",
    ]

    display_floor = positive_weights.min() * 0.7

    for field in fields:
        line = panel[
            panel.analysis_group.eq(field)
        ].sort_values("lag")

        if line.empty:
            continue

        positive = line.weight.gt(0)

        ax.plot(
            line.loc[positive, "lag"],
            line.loc[positive, "weight"],
            color=colors[field],
            marker=markers[field],
            linewidth=2.2,
            markersize=7,
            alpha=0.9,
        )

        nonpositive = line[~positive]

        if not nonpositive.empty:
            ax.scatter(
                nonpositive.lag,
                np.full(len(nonpositive), display_floor),
                color=colors[field],
                marker="x",
                s=60,
                linewidth=2,
                zorder=5,
            )

    ax.set_yscale("log")
    ax.set_xticks(range(2, 7))
    ax.set_xlabel("Historical lag $k$ (years)")
    ax.set_ylabel(
        r"Stage-averaged raw AR memory weight $\bar{w}_k$"
    )
    ax.set_title(equation_titles[equation])
    ax.grid(True, which="major", alpha=0.22)
    ax.grid(True, which="minor", alpha=0.08)

field_handles = [
    Line2D(
        [0],
        [0],
        color=colors[field],
        marker=markers[field],
        linewidth=2.2,
        markersize=7,
        label=field,
    )
    for field in fields
]

fig.legend(
    handles=field_handles,
    title="Field / dataset",
    loc="center left",
    bbox_to_anchor=(0.89, 0.5),
    frameon=False,
)

fig.suptitle(
    "Raw unrestricted-AR memory decay by field",
    fontsize=17,
    fontweight="bold",
)

fig.text(
    0.5,
    0.015,
    "weighted by stage duration. Straight lines suggest "
    "geometric decay; steeper slopes suggest faster forgetting",
    ha="center",
    fontsize=10,
)

fig.tight_layout(
    rect=[0, 0.07, 0.88, 0.93]
)

figure_path = (
    OUTPUT
    / "raw_ar_decay_weights_log_scale_by_field_stage_averaged.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(figure_path)

/Users/slmagid/Desktop/SHL_dblp_comparable/f_compare/output/raw_ar_decay_weights_log_scale_by_field_stage_averaged.png


/var/folders/h2/vczqtqf12sxc_lw7b1z773cr0000gn/T/ipykernel_15830/531771663.py:203: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
